# TAE-IA · Módulo 6 · L24 — Audio Backend: the demo

> **Generated from `M6_L24_L25_audio_vision_app.ipynb` by `tools/lab_builders/split_l24_demo.py`.**
> Edit the merged notebook and re-run the script; do not hand-edit this file.

**Instructor reference. This notebook is *not* handed to students, and it is not typed live.**
You run it; they watch. What they build afterwards is a *different* app — see
`M6_L24_skeleton_audio.ipynb`, which is the file they open.

| | |
|---|---|
| **Session** | L24 — the audio half |
| **Your slot** | ~20 min, sections 1–6, straight through |
| **GPU** | T4 (Colab) |
| **Models** | ESC-50 checkpoint 20 MB · Whisper small 0.5 GB · XTTS-v2 1.8 GB · MusicGen-small on demand |
| **Monday** | `M6_L24_L25_audio_vision_app.ipynb` — the full multimodal app, demoed in L25 |

### What this demo is for

The students' app shares **no code** with this one: it tags with CLAP and CLIP embeddings where
this one describes with BLIP and XTTS. Say that out loud, or several of them will spend twenty
minutes trying to copy from it.

What they *do* need from the demo is one thing, and it is in section 6:

```python
gr.Audio(type='numpy')   # the callback receives (sr, wav) — rate FIRST
```

`librosa.load` returns `(wav, sr)`. Gradio returns `(sr, wav)`. Unpacked backwards nothing
raises — you resample to the length of the array and listen to noise. Every student writes a
`gr.Audio` callback today, so this is the cell that saves the room an hour.

### ⚠ Do not use *Run all*

Section 6 launches a Gradio app that the **next** cell shuts down with `demo.close()` —
deliberate, since two apps cannot share a port. *Run all* leaves you with a dead interface and
*"No interface is running right now"*. Run section 6 cell by cell: launch, look at what it
printed, discuss it, *then* close it.

### Inputs live on Drive, not on the internet

Everything here reads from `TAE_IA_M6/inputs/` — the same shared folder
`M6_L11_vision_app.ipynb` uses. `list_inputs()` prints what is there; `upload_inputs()` opens a
file picker. Nothing in the teaching path makes a network call, because Wikimedia rate-limits:
25 students fetching one URL at once earns the room an HTTP 429.

### ⚠ Before class: two files must already be on Drive

Both live in `MyDrive/TAE_IA_M6/`, and neither can be re-downloaded in the room:

- **`ESC50_best.pth`** — the L17 checkpoint. Without it section 2 raises `FileNotFoundError`
  and the demo stops at its second cell.
- **`ESC-50-master/meta/esc50.csv`** — without it the labels fall back to `class-37` instead of
  `rain`, and every narrated sentence is meaningless.

Neither is needed by the students' own build — their app uses CLAP, which has no checkpoint.
These two are for **this notebook only**.

### Talking points that earn their minute

- **Section 2** — `vram()` after each load. Say the number out loud each time. *"That is the
  answer to 'will my project fit'."* The slide deck's table is these numbers.
- **Section 3** — `coerce_audio`. This is the one function that also appears, unchanged, in the
  students' skeleton. Point at it and say so: it is given to them because it is where silent
  failure lives.
- **Section 4** — the `conf < 0.45` branch. Ask the room what a 3%-confidence label should do.
  They say "return it anyway"; that is how you narrate a guess as a fact. Their spec requires
  the same branch, with a floor they measure themselves.
- **Section 5** — the silence case. The amplitude assertion fires and the clip never reaches
  Whisper. Comment that one line out, re-run, and let the room watch Whisper invent a sentence
  out of digital silence — then put it back. Best demonstration in Track B of why assertions
  beat vigilance.
- **Section 6** — the trap. Have them write the tuple order down before you close the app.

### The install, and why it looks the way it does

Three things in section 1 are not optional on current Colab (Python 3.13 / transformers 5 /
torch 2.11), all verified on a T4 on 2026-09-18:

- **`coqui-tts`, not `TTS`.** The original package caps at Python < 3.12 and will not install.
- **The `[codec]` extra.** torch >= 2.9 does audio I/O through torchcodec.
- **The `isin_mps_friendly` shim** in section 2, before `from TTS.api import ...`. transformers
  5.1 removed the helper coqui-tts still imports (idiap/coqui-ai-TTS#558, open).

`coqui-tts` 0.27.5 left `huggingface_hub` at 1.29.0 and `pip check` came back clean. Do not
upgrade hub *after* the install: that is what breaks `import TTS`.

> **`ultralytics` and `gradio` are installed here although day 1 uses only `gradio`.** The
> install cell is shared with Monday's notebook, and paying two seconds now beats stopping to
> pip in front of the room.

---

## 1 — Setup and model cache
> Weights on the runtime disk, outputs on Drive. Re-run this every session.

In [1]:
# ============================================================
# 1. CONFIGURACIÓN DEL ENTORNO COLAB + GOOGLE DRIVE
# ============================================================

import os
import sys
import gc
import time
import random
import textwrap

import numpy as np
import torch

# ------------------------------------------------------------
# Google Drive
# ------------------------------------------------------------
from google.colab import drive

DRIVE_MOUNT = '/content/drive'

try:
    # Si ya está montado, no forzar una nueva autenticación
    if os.path.exists(f'{DRIVE_MOUNT}/MyDrive'):
        print('✓ Google Drive ya está montado.')
    else:
        print('Montando Google Drive...')
        drive.mount(DRIVE_MOUNT)
        print('✓ Google Drive montado correctamente.')

except Exception as e:
    print('\nERROR AL MONTAR GOOGLE DRIVE')
    print(type(e).__name__, ':', e)
    print('\nSi aparece "credential propagation was unsuccessful":')
    print('1. Desconecta y elimina el entorno de ejecución.')
    print('2. Vuelve a conectarte.')
    print('3. Ejecuta nuevamente esta celda.')
    raise

# ------------------------------------------------------------
# Rutas del proyecto
# ------------------------------------------------------------
DRIVE_ROOT = '/content/drive/MyDrive/TAE_IA_M6'

OUTPUT_DIR = f'{DRIVE_ROOT}/L24_output'
INPUT_DIR  = f'{DRIVE_ROOT}/inputs'

for d in (OUTPUT_DIR, INPUT_DIR):
    os.makedirs(d, exist_ok=True)

print(f'✓ DRIVE_ROOT : {DRIVE_ROOT}')
print(f'✓ OUTPUT_DIR : {OUTPUT_DIR}')
print(f'✓ INPUT_DIR  : {INPUT_DIR}')

# ------------------------------------------------------------
# Modelos y cachés en disco local del runtime
# ------------------------------------------------------------
MODEL_CACHE = '/content/models'

os.makedirs(MODEL_CACHE, exist_ok=True)

os.environ['HF_HOME']          = MODEL_CACHE
os.environ['TORCH_HOME']       = MODEL_CACHE
os.environ['XDG_CACHE_HOME']   = MODEL_CACHE
os.environ['TTS_HOME']         = f'{MODEL_CACHE}/tts'
os.environ['YOLO_CONFIG_DIR']  = MODEL_CACHE
os.environ['COQUI_TOS_AGREED'] = '1'

print(f'✓ MODEL_CACHE: {MODEL_CACHE}')

# ------------------------------------------------------------
# Verificación GPU
# ------------------------------------------------------------
if not torch.cuda.is_available():
    raise SystemExit(
        'No GPU disponible.\n'
        'Ve a: Entorno de ejecución > Cambiar tipo de entorno de ejecución > T4 GPU'
    )

# ------------------------------------------------------------
# Semillas reproducibles
# ------------------------------------------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Información GPU / VRAM
# ------------------------------------------------------------
def vram(tag=''):
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  VRAM {used:5.2f} / {total:.1f} GB   {tag}')

print('\nGPU:')
print(torch.cuda.get_device_name(0))

vram('empty')

print('\n✓ ENTORNO L24 INICIALIZADO CORRECTAMENTE')

Montando Google Drive...
Mounted at /content/drive
✓ Google Drive montado correctamente.
✓ DRIVE_ROOT : /content/drive/MyDrive/TAE_IA_M6
✓ OUTPUT_DIR : /content/drive/MyDrive/TAE_IA_M6/L24_output
✓ INPUT_DIR  : /content/drive/MyDrive/TAE_IA_M6/inputs
✓ MODEL_CACHE: /content/models

GPU:
Tesla T4
  VRAM  0.00 / 15.6 GB   empty

✓ ENTORNO L24 INICIALIZADO CORRECTAMENTE


In [2]:
# `TTS` (the original Coqui package) caps at Python < 3.12 and cannot install on Colab any
# more. `coqui-tts` is the maintained fork; the [codec] extra pulls in torchcodec, which
# torch >= 2.9 needs for audio I/O. Same line L21 and L22 use.
# ultralytics is day 2's YOLO - installed here so day 2 never stops to pip.
!pip install -q openai-whisper ultralytics gradio librosa soundfile
!pip install -q "coqui-tts[codec]"
# Measured on Colab 2026-09-18: hub stays at 1.29.0, pip check is clean, and day 2's
# snapshot_download + BLIP work fine afterwards. (An earlier Colab image shipped hub 1.30,
# which coqui-tts DID downgrade - if a future image does that again, day 2 is what breaks.)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 21.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.3/997.3 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 639.3/639.3 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

## 2 — Load the audio models
> Three resident, one on demand. Watch the VRAM line grow, and note what we *do not* load.

In [3]:
import librosa, soundfile as sf
import whisper
import torch.nn as nn
from torchvision import models as tvm, transforms as T
from PIL import Image

# --- 1. Your ESC-50 classifier from L17 (20 MB, ~10 ms) ---
CKPT = f'{DRIVE_ROOT}/ESC50_best.pth'
if not os.path.exists(CKPT):
    raise FileNotFoundError(
        f'{CKPT} not found. This is the one artefact you cannot re-download - '
        'it is the checkpoint you trained in L17.')

def build_classifier(num_classes=50):
    m = tvm.efficientnet_b0(weights=None)                 # weights come from the ckpt
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    return m

clf = build_classifier().to('cuda').eval()
state = torch.load(CKPT, map_location='cuda')   # L17 saves a bare state_dict
clf.load_state_dict(state['model_state_dict'] if 'model_state_dict' in state else state)

# Label names: from the ESC-50 metadata if it survived, else numeric.
ESC50_META = f'{DRIVE_ROOT}/ESC-50-master/meta/esc50.csv'
if os.path.exists(ESC50_META):
    import pandas as pd
    _m = pd.read_csv(ESC50_META).drop_duplicates('target').sort_values('target')
    ESC50_LABELS = _m['category'].tolist()
else:
    ESC50_LABELS = [f'class-{i}' for i in range(50)]
    print('  ! esc50.csv not on Drive - labels will be numeric')
print(f'classifier: {len(ESC50_LABELS)} classes'); vram('after classifier')

# --- 2. Whisper small (0.5 GB download, ~2 GB VRAM, seconds per clip) ---
asr = whisper.load_model('small', download_root=MODEL_CACHE)
vram('after Whisper')

# --- 3. XTTS-v2 (1.8 GB, ~2 GB VRAM, seconds per sentence) ---
# coqui-tts 0.27 still imports a helper transformers 5.1 removed. One line, before the
# import, or `from TTS.api import ...` raises ImportError. Upstream: idiap/coqui-ai-TTS#558.
import transformers.pytorch_utils as _pu
if not hasattr(_pu, 'isin_mps_friendly'):
    _pu.isin_mps_friendly = torch.isin

from TTS.api import TTS as CoquiTTS
tts = CoquiTTS('tts_models/multilingual/multi-dataset/xtts_v2').to('cuda')
vram('all three resident')

# --- 4. MusicGen: NOT loaded. It is ~4 GB and used on one branch only.
#        Loaded on demand in `generate()` below, and freed straight after.
#        This is the decision that makes day 2's merge fit on one T4.

  ! esc50.csv not on Drive - labels will be numeric
classifier: 50 classes
  VRAM  0.03 / 15.6 GB   after classifier


100%|████████████████████████████████████████| 461M/461M [00:03<00:00, 155MiB/s]


  VRAM  1.00 / 15.6 GB   after Whisper


100%|██████████| 1.87G/1.87G [00:28<00:00, 65.8MiB/s]
4.37kiB [00:00, 6.86MiB/s]
361kiB [00:00, 98.2MiB/s]
100%|██████████| 32.0/32.0 [00:00<00:00, 59.7kiB/s]
100%|██████████| 7.75M/7.75M [00:00<00:00, 24.8MiB/s]
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 255), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 255), got 50256. This may result in unexpected behavior.
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 607), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 607), got 50256. This may result in unexpected behavior.


  VRAM  2.87 / 15.6 GB   all three resident


## 3 — Helpers: the input contract
> `coerce_audio` is `coerce_image`'s twin. Tomorrow Gradio hands us `(sr, wav)` — rate first.

In [4]:
import requests
from io import BytesIO

# The rates every stage wants. Four different numbers in one pipeline.
CLASSIFIER_SR = 22050    # L16: TARGET_SR
ASR_SR        = 16000    # Whisper resamples internally, but be explicit
TTS_SR        = 24000    # XTTS-v2 output
MAX_SECONDS   = 60       # a 30-min upload is the #1 cause of a demo-day stall
MIN_SECONDS   = 0.5

# --- Inputs come from Drive, not from the internet -------------------------
# Same shared folder as L11/L12. Upload clips once; no lesson ever depends on a
# website being reachable. (Wikimedia rate-limits a whole classroom at once.)

def list_inputs():
    """What is actually in the shared folder right now."""
    names = sorted(f for f in os.listdir(INPUT_DIR) if not f.startswith('.'))
    print(f'{INPUT_DIR}  ({len(names)} files)')
    for n in names:
        print(f'  {os.path.getsize(os.path.join(INPUT_DIR, n))/1e6:6.2f} MB  {n}')
    return names

def upload_inputs():
    """Pick files from your machine; they land in Drive and stay there."""
    from google.colab import files
    for fname, data in files.upload().items():
        with open(os.path.join(INPUT_DIR, fname), 'wb') as f:
            f.write(data)
        print(f'  saved {fname}')

def load_audio(name_or_path, sr=None):
    """A name in the inputs folder, or any path -> (wav float32 mono, sr)."""
    path = name_or_path
    if not os.path.isabs(path):
        path = os.path.join(INPUT_DIR, name_or_path)
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'{name_or_path} is not in {INPUT_DIR}. '
            'Run upload_inputs() and choose it, or copy it into that folder in Drive.')
    wav, got_sr = librosa.load(path, sr=sr, mono=True)   # sr=None keeps the original
    return wav.astype('float32'), got_sr

# The reference voice for XTTS: the inputs folder first, then whatever L21 wrote.
REF_WAV = next((p for p in (f'{INPUT_DIR}/reference.wav',
                            f'{DRIVE_ROOT}/tts_output/reference.wav')
                if os.path.exists(p)), None)
REF_SPEAKER = None if REF_WAV else 'Ana Florence'       # built-in studio voice
print('XTTS reference:', REF_WAV or f'built-in {REF_SPEAKER}')

def coerce_audio(wav, sr, target_sr):
    """Anything a user can hand us -> (float32 mono at target_sr), or ValueError."""
    if wav is None:
        raise ValueError('No audio received. Please upload a clip.')
    wav = np.asarray(wav)
    if wav.dtype.kind in 'iu':                       # gradio hands back int16
        wav = wav.astype('float32') / np.iinfo(wav.dtype).max
    wav = wav.astype('float32')
    if wav.ndim > 1:                                 # to mono, either layout
        wav = wav.mean(axis=0) if wav.shape[0] < wav.shape[1] else wav.mean(axis=1)
    if wav.size < sr * MIN_SECONDS:
        raise ValueError(f'Clip too short ({wav.size/sr:.2f} s). Use at least {MIN_SECONDS} s.')
    if wav.size > sr * MAX_SECONDS:
        wav = wav[:int(sr * MAX_SECONDS)]            # truncate, and the caller says so
    if np.abs(wav).max() < 1e-4:
        raise ValueError('That clip is silent. Whisper would invent words for it.')
    if sr != target_sr:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=target_sr)
    return wav.astype('float32'), target_sr

# --- the L16 spectrogram, reproduced EXACTLY. Any drift here breaks the checkpoint. ---
N_FFT, HOP_LENGTH, N_MELS, FMAX, IMG_SIZE = 2048, 512, 128, 8000, 128
TARGET_SAMPLES = CLASSIFIER_SR * 5

_tf = T.Compose([
    T.Resize((224, 224)),                                  # L17 resized to 224, not 128
    T.ToTensor(),
    T.Lambda(lambda x: x.repeat(3, 1, 1)),                 # grayscale -> 3 channels
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def mel_image(wav):
    """5 s at 22050 Hz -> the same 128x128 grayscale PIL image L16 saved as PNG."""
    y = wav[:TARGET_SAMPLES] if len(wav) > TARGET_SAMPLES else \
        np.pad(wav, (0, TARGET_SAMPLES - len(wav)))
    S    = librosa.feature.melspectrogram(y=y, sr=CLASSIFIER_SR, n_fft=N_FFT,
                                          hop_length=HOP_LENGTH, n_mels=N_MELS, fmax=FMAX)
    S_db = librosa.power_to_db(S, ref=np.max)
    S_n  = ((S_db - S_db.min()) / (S_db.max() - S_db.min() + 1e-8) * 255).astype(np.uint8)
    S_n  = np.flipud(S_n)                                  # low freq at bottom
    return Image.fromarray(S_n).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)  # 2-D uint8 -> 'L'

def classify(wav, sr, topk=3):
    """(wav, sr) -> [(label, confidence)] sorted. Assumes sr == CLASSIFIER_SR."""
    assert sr == CLASSIFIER_SR, f'{sr} != {CLASSIFIER_SR} - coerce first'
    x = _tf(mel_image(wav)).unsqueeze(0).to('cuda')
    with torch.no_grad():
        p = torch.softmax(clf(x)[0], dim=-1)
    conf, idx = p.topk(topk)
    return [(ESC50_LABELS[i], float(c)) for c, i in zip(conf, idx)]

def transcribe(wav, sr, language=None):
    """(wav, sr) -> text. Whisper wants float32 at 16 kHz."""
    assert sr == ASR_SR, f'{sr} != {ASR_SR} - coerce first'
    out = asr.transcribe(wav, language=language, fp16=True)
    return out['text'].strip()

def narrate(text, language='es'):
    """text -> path to a 24 kHz WAV. The one stage both branches will share tomorrow."""
    out_wav = f'{OUTPUT_DIR}/narration.wav'
    kw = {'speaker_wav': REF_WAV} if REF_SPEAKER is None else {'speaker': REF_SPEAKER}
    tts.tts_to_file(text=text, language=language, file_path=out_wav, **kw)
    return out_wav

def generate(prompt, seconds=5):
    """text -> (wav, sr). Loads MusicGen, uses it, frees it. ~15 s per 10 s of audio."""
    from transformers import AutoProcessor, MusicgenForConditionalGeneration
    proc = AutoProcessor.from_pretrained('facebook/musicgen-small')
    mg   = MusicgenForConditionalGeneration.from_pretrained(
               'facebook/musicgen-small', torch_dtype=torch.float16).to('cuda')
    try:
        inp = proc(text=[prompt], padding=True, return_tensors='pt').to('cuda')
        with torch.no_grad():
            out = mg.generate(**inp, do_sample=True, max_new_tokens=int(seconds * 50))
        return out[0, 0].cpu().float().numpy(), mg.config.audio_encoder.sampling_rate
    finally:
        del mg, proc                      # the residency policy, in three lines
        gc.collect(); torch.cuda.empty_cache()

XTTS reference: built-in Ana Florence


## 4 — The audio pipeline
> The conditional pattern: what we heard decides what happens next. This is the whole backend.

In [5]:
CONF_FLOOR = 0.45          # below this we refuse to name the sound

def analyse_audio(wav, sr):
    """Detect what kind of clip this is, and describe it. Returns a dict, with timings."""
    t = {}

    # Branch on content, not on file type. Speech goes to Whisper, everything else
    # to the classifier. A cheap heuristic first, then the expensive model.
    w16, _ = coerce_audio(wav, sr, ASR_SR)
    t0 = time.time(); text = transcribe(w16, ASR_SR); t['whisper'] = time.time() - t0

    if len(text.split()) >= 3:
        kind, label, conf = 'speech', None, None
        summary = f'Escuché voz. Dice: {text}'
    else:
        w22, _ = coerce_audio(wav, sr, CLASSIFIER_SR)
        t0 = time.time(); top = classify(w22, CLASSIFIER_SR); t['classifier'] = time.time() - t0
        label, conf = top[0]
        kind = 'sound'
        # A 50-class model cannot say "not one of mine". The threshold says it for us.
        summary = (f'Este sonido parece {label}.' if conf >= CONF_FLOOR
                   else 'No pude identificar este sonido.')

    t['total'] = sum(t.values())
    return {'kind': kind, 'text': text, 'label': label, 'confidence': conf,
            'summary': summary, 'timings': t}

def run_audio(x, sr=None, language='es'):
    """The one function the UI will call tomorrow. Anything in, never raises."""
    try:
        if isinstance(x, str):
            wav, sr = load_audio(x, sr=None)
        else:
            wav = x
            if sr is None:
                raise ValueError('No sample rate given. Audio is always (wav, sr).')
        wav, sr = coerce_audio(wav, sr, sr)          # validates; does not resample yet
    except ValueError as e:
        return None, None, str(e)

    try:
        a       = analyse_audio(wav, sr)
        out_wav = narrate(a['summary'], language=language)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return None, None, 'The GPU ran out of memory. Try a shorter clip.'
    except Exception as e:
        print(f'[run_audio] {type(e).__name__}: {e}')                  # for us
        return None, None, 'Something went wrong. Try another clip.'   # for them

    stages = ', '.join(f'{k} {v:.2f}s' for k, v in a['timings'].items() if k != 'total')
    report = (f"heard   : {a['kind']}"
              + (f" ({a['label']}, {a['confidence']:.2f})" if a['label'] else '') + '\n'
              f"said    : {a['summary']}\n"
              f"time    : {stages}")
    return a['summary'], out_wav, report

## 5 — Test the backend before it ever meets a user
> Day 1 ends here. Every hostile input returns a sentence, not a traceback.

In [6]:
# Real clips if you have put any in the shared folder; a synthetic one always.
rng   = np.random.default_rng(SEED)
noise = (0.2 * rng.standard_normal(CLASSIFIER_SR * 5)).astype('float32')

print('--- what is in the inputs folder ---')
clips = [f for f in list_inputs() if f.lower().endswith(('.wav', '.mp3', '.ogg', '.flac'))]

print('\n--- real runs ---')
for name in clips[:3]:
    wav, sr = load_audio(name)
    _s, _w, report = run_audio(wav, sr=sr)
    print(f'{name}:'); print(report)

if not clips:
    print('(no clips in the folder - upload_inputs() to add some)')
summary, wav_path, report = run_audio(noise, sr=CLASSIFIER_SR)
print('synthetic noise:'); print(report)

# Hostile inputs: none of these may raise.
print('\n--- hostile inputs ---')
cases = [
    ('None',        None,                                              CLASSIFIER_SR),
    ('silence',     np.zeros(CLASSIFIER_SR * 5, dtype='float32'),      CLASSIFIER_SR),
    ('stereo',      np.stack([noise, noise], axis=-1),                 CLASSIFIER_SR),
    ('int16',       (noise * 32767).astype('int16'),                   CLASSIFIER_SR),
    ('wrong sr',    noise,                                             44100),
    ('0.1 s',       noise[:int(CLASSIFIER_SR * 0.1)],                  CLASSIFIER_SR),
    ('30 minutes',  np.tile(noise, 360),                               CLASSIFIER_SR),
]
for tag, bad, sr in cases:
    _s, _w, msg = run_audio(bad, sr=sr)
    print(f'  {tag:12s} -> {msg.splitlines()[0][:64]}')

vram('peak')
print(f"peak {torch.cuda.max_memory_allocated()/1e9:.2f} GB of "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print('\nThis table is what L25 plans against. Write it down.')

--- what is in the inputs folder ---
/content/drive/MyDrive/TAE_IA_M6/inputs  (88 files)
    0.44 MB  Autoclave  esterilizacion.com (11) (1).png
    0.44 MB  Autoclave  esterilizacion.com (11).png
    0.31 MB  Bomba de infusión medicina equipo médico bomba peristáltica.com (3) (1).png
    0.31 MB  Bomba de infusión medicina equipo médico bomba peristáltica.com (3).png
    0.00 MB  CREDITOS.txt
    5.88 MB  Cama de hospital.com (10) (1).png
    5.88 MB  Cama de hospital.com (10).png
    0.22 MB  Electrocardiograma.com (7) (1).png
    0.22 MB  Electrocardiograma.com (7).png
    1.20 MB  Equipo de resonancia magnetica.com (2) (1).png
    1.20 MB  Equipo de resonancia magnetica.com (2).png
    0.02 MB  LICENSE_CC-BY-SA-4.0.txt
    0.30 MB  Monitorización de signos vitales, monitores informáticos, equipos médicos, presión arterial., vigilancia, vital, señales.com (4) (1).png
    0.30 MB  Monitorización de signos vitales, monitores informáticos, equipos médicos, presión arteri

## 6 — Hello Gradio, and the trap — audio edition
> Same lesson as L12, new type. Look at what it actually handed us before trusting it.
>
> **Run this cell on its own, not with *Run all*.** The next cell closes this app on purpose.

In [7]:
import gradio as gr
print('gradio', gr.__version__)

def what_did_i_get(audio):
    if audio is None:
        return 'Got None.'
    a, b = audio
    return (f'tuple of ({type(a).__name__}, {type(b).__name__})\n'
            f'  first  = {a}  <- this is the SAMPLE RATE\n'
            f'  second = ndarray shape={b.shape} dtype={b.dtype}\n\n'
            'librosa.load returns (wav, sr). Gradio returns (sr, wav).\n'
            'Unpack it backwards and nothing raises - you just resample to the\n'
            'array length and hear noise.')

demo = gr.Interface(fn=what_did_i_get,
                    inputs=gr.Audio(type='numpy'),
                    outputs=gr.Textbox(lines=8),
                    title='What did I just get?')
demo.launch(share=True, quiet=True)

gradio 6.26.0
* Running on public URL: https://89a53308a08f649a9e.gradio.live


In [ ]:
demo.close()      # always close before relaunching, or you leak ports